In [0]:
CREATE OR REPLACE PROCEDURE
dbw_healthcare_de_dev.etl.usp_update_file_stage
(
    IN p_file_id STRING,
    IN p_adf_run_id STRING,
    IN p_completed_stage STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

    UPDATE dbw_healthcare_de_dev.etl.file_control
    SET
        processing_stage =
            CASE
                WHEN p_completed_stage = 'BRONZE' THEN 'SILVER'
                WHEN p_completed_stage = 'SILVER' THEN 'GOLD'
                WHEN p_completed_stage = 'GOLD' THEN 'COMPLETED'
            END,

        pipeline_status =
            CASE
                WHEN p_completed_stage = 'GOLD' THEN 'COMPLETED'
                ELSE 'IN_PROGRESS'
            END,

        processing_end_ts =
            CASE
                WHEN p_completed_stage = 'GOLD'
                    THEN CURRENT_TIMESTAMP()
                ELSE processing_end_ts
            END,

        last_updated_ts = CURRENT_TIMESTAMP()

    WHERE file_id = p_file_id
      AND adf_run_id = p_adf_run_id
      AND pipeline_status = 'IN_PROGRESS'

      AND processing_stage = p_completed_stage

      AND p_completed_stage IN ('BRONZE', 'SILVER', 'GOLD')

      AND EXISTS (
          SELECT 1
          FROM dbw_healthcare_de_dev.etl.file_stage_audit a
          WHERE a.file_id = p_file_id
            AND a.adf_run_id = p_adf_run_id
            AND a.processing_stage = p_completed_stage
            AND a.stage_status = 'SUCCESS'
            AND a.stage_end_ts IS NOT NULL
      );

END;